In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image

import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install sentence-transformers torch matplotlib scipy -q

# Experiment Overview

Experiment 9b — Probe 4 in Projected Space + Production-Cost Lewis Game

This resolves TWO explicit open questions left dangling in the report's Exp 9.

PART 1 — Probe 4 (cross-linguistic RSA) in the projected space.
  Exp 9 Probe 4 verdict was NOT_DETECTED*, with an asterisk:
  "The key open question is whether D_projected-universal — computed in the
   Phase 1 space rather than the raw centroid space — reverses the verdict.
   If the projection increases the minimum D_universal-D_lang correlation above
   the maximum cross-language correlation, Probe 4 becomes EMERGENT."
  We compute it.

PART 2 — Lewis signalling game WITH a production cost.
  Exp 9 Phase 3 found symbol entropy RISES (anti-efficient encoding; agents
  exploit full channel capacity) because there is no production cost.
  "Adding a production cost penalty is the direct fix, and would recover Zipfian
   symbol distribution (Galke & Raviv 2024), completing the circuit:
   discreteness (Exp 1) → optimal opacity (Exp 3) → production pressure."
  We add an entropy/length penalty and test whether Zipfian structure emerges.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, linregress
import warnings
warnings.filterwarnings('ignore')

device = 'cpu'
model = SentenceTransformer('LaBSE', device='cpu')
print('LaBSE loaded')

# ── 56-concept vocabulary across 5 languages (same as Exp 9) ─────────────────
CONCEPTS = [
    'water','fire','earth','sky','wind','stone','river','mountain','forest','child',
    'elder','friend','enemy','time','life','death','peace','war','hope','dream',
    'change','love','fear','trust','joy','pain','give','take','speak','think',
    'find','lose','build','break','eat','feel','sleep','run','light','dark',
    'teach','learn','kill','grow','fall','rise','sing','cry',
    'knife','hand','voice','home','silence','memory','freedom','beauty',
]
SPEAKER_WEIGHTS = {'en':1500, 'zh':1100, 'es':560, 'ar':380, 'ru':260}
LANG_VOCAB = {
    'en': CONCEPTS,
    'zh': ['水','火','土','天空','风','石头','河流','山','森林','孩子',
           '老人','朋友','敌人','时间','生命','死亡','和平','战争','希望','梦想',
           '改变','爱','恐惧','信任','喜悦','痛苦','给','拿','说话','思考',
           '找到','失去','建造','破坏','吃','感觉','睡觉','跑','光','暗',
           '教','学习','杀','生长','落下','升起','唱歌','哭',
           '刀','手','声音','家','沉默','记忆','自由','美丽'],
    'es': ['agua','fuego','tierra','cielo','viento','piedra','río','montaña','bosque','niño',
           'anciano','amigo','enemigo','tiempo','vida','muerte','paz','guerra','esperanza','sueño',
           'cambio','amor','miedo','confianza','alegría','dolor','dar','tomar','hablar','pensar',
           'encontrar','perder','construir','romper','comer','sentir','dormir','correr','luz','oscuridad',
           'enseñar','aprender','matar','crecer','caer','subir','cantar','llorar',
           'cuchillo','mano','voz','hogar','silencio','memoria','libertad','belleza'],
    'ar': ['ماء','نار','أرض','سماء','ريح','حجر','نهر','جبل','غابة','طفل',
           'مسن','صديق','عدو','وقت','حياة','موت','سلام','حرب','أمل','حلم',
           'تغير','حب','خوف','ثقة','فرح','ألم','أعطى','أخذ','تكلم','فكر',
           'وجد','فقد','بنى','كسر','أكل','شعر','نوم','ركض','ضوء','ظلام',
           'علم','تعلم','قتل','نما','سقط','صعد','غنى','بكى',
           'سكين','يد','صوت','بيت','صمت','ذاكرة','حرية','جمال'],
    'ru': ['вода','огонь','земля','небо','ветер','камень','река','гора','лес','ребёнок',
           'старик','друг','враг','время','жизнь','смерть','мир','война','надежда','мечта',
           'изменение','любовь','страх','доверие','радость','боль','давать','брать','говорить','думать',
           'найти','потерять','строить','ломать','есть','чувствовать','спать','бежать','свет','тьма',
           'учить','учиться','убить','расти','падать','подниматься','петь','плакать',
           'нож','рука','голос','дом','тишина','память','свобода','красота'],
}

print('Embedding multilingual vocabulary...')
df_rows = []
for lang, words in LANG_VOCAB.items():
    embs = model.encode(words, normalize_embeddings=True)
    for concept, word, emb in zip(CONCEPTS, words, embs):
        df_rows.append({'lang': lang, 'concept': concept, 'word': word, 'emb': emb})
df = pd.DataFrame(df_rows)

centroids = {}
for concept in CONCEPTS:
    sub = df[df['concept'] == concept]
    w = np.array([SPEAKER_WEIGHTS[l] for l in sub['lang']], dtype=float); w /= w.sum()
    c = (np.stack(sub['emb'].values) * w[:, None]).sum(0); c /= np.linalg.norm(c)
    centroids[concept] = c
centroid_matrix = np.stack([centroids[c] for c in CONCEPTS])

# ── Phase 1 projection (same as Exp 9 / Exp 10) ──────────────────────────────
ANALOGY_PAIRS = [
    ('love','fear','joy','pain'),('life','death','peace','war'),
    ('give','take','build','break'),('light','dark','hope','pain'),
    ('friend','enemy','peace','war'),('find','lose','give','take'),
    ('rise','fall','build','break'),('sing','cry','joy','pain'),
    ('teach','learn','give','take'),
]
class Projection(nn.Module):
    def __init__(self, d=768):
        super().__init__(); self.W = nn.Linear(d, d, bias=False); nn.init.eye_(self.W.weight)
    def forward(self, x): return F.normalize(self.W(x), dim=-1)

print('Training Phase 1 projection...')
proj = Projection(); opt = torch.optim.Adam(proj.parameters(), lr=5e-4)
X = torch.tensor(centroid_matrix, dtype=torch.float32)
for epoch in range(800):
    P = proj(X)
    L_prox = 1.0 - (P * X).sum(-1).mean()
    L_comp = torch.tensor(0.0); n = 0
    for a,b,c,d_ in ANALOGY_PAIRS:
        ia,ib,ic,id_ = [CONCEPTS.index(x) for x in (a,b,c,d_)]
        pred = F.normalize(P[ia]-P[ib]+P[ic], dim=0)
        L_comp = L_comp + (1.0 - (pred*P[id_]).sum()); n += 1
    L_comp = L_comp / n
    loss = L_prox + 2.0 * L_comp
    opt.zero_grad(); loss.backward(); opt.step()
proj.eval()
with torch.no_grad():
    projected_matrix = proj(X).numpy()
print('Projection trained.')

## PART 1 — Probe 4 in raw vs projected space

In [ ]:
print('\n' + '═'*60)
print('PART 1 — Probe 4: Cross-linguistic RSA (raw vs projected)')
print('═'*60)

triu = np.triu_indices(len(CONCEPTS), k=1)

def lang_dist_matrix(lang):
    sub = df[df['lang']==lang].set_index('concept').loc[CONCEPTS]
    embs = np.stack(sub['emb'].values)
    return squareform(pdist(embs, 'cosine'))

lang_dists = {l: lang_dist_matrix(l) for l in LANG_VOCAB}

def probe4(universal_matrix, label):
    D_univ = squareform(pdist(universal_matrix, 'cosine'))[triu]
    # universal vs each language
    univ_corrs = {}
    for l in LANG_VOCAB:
        rho, _ = spearmanr(D_univ, lang_dists[l][triu])
        univ_corrs[l] = rho
    # inter-language
    langs = list(LANG_VOCAB.keys())
    inter = []
    for i in range(len(langs)):
        for j in range(i+1, len(langs)):
            rho, _ = spearmanr(lang_dists[langs[i]][triu], lang_dists[langs[j]][triu])
            inter.append(rho)
    min_univ = min(univ_corrs.values())
    max_inter = max(inter)
    mean_univ = np.mean(list(univ_corrs.values()))
    mean_inter = np.mean(inter)
    print(f'\n  [{label}]')
    print(f'    min(D_universal–D_lang) ρ = {min_univ:.4f}')
    print(f'    max(cross-language)     ρ = {max_inter:.4f}')
    print(f'    mean univ–lang ρ = {mean_univ:.4f}  |  mean inter-lang ρ = {mean_inter:.4f}')
    verdict = 'EMERGENT' if min_univ > max_inter else 'NOT_DETECTED'
    print(f'    VERDICT: {verdict}')
    return {'label': label, 'univ_corrs': univ_corrs, 'inter': inter,
            'min_univ': min_univ, 'max_inter': max_inter,
            'mean_univ': mean_univ, 'mean_inter': mean_inter, 'verdict': verdict}

raw_p4 = probe4(centroid_matrix, 'raw centroid space')
proj_p4 = probe4(projected_matrix, 'Phase 1 projected space')

print('\n  → Does the projection FLIP the verdict?')
if raw_p4['verdict'] == 'NOT_DETECTED' and proj_p4['verdict'] == 'EMERGENT':
    print('    YES — Probe 4 becomes EMERGENT in the projected space. Asterisk resolved.')
elif raw_p4['verdict'] == proj_p4['verdict']:
    print(f'    NO — both spaces give {raw_p4["verdict"]}. But check whether the projection')
    print('    at least narrows the gap (min_univ − max_inter):')
    print(f'      raw gap:       {raw_p4["min_univ"] - raw_p4["max_inter"]:+.4f}')
    print(f'      projected gap: {proj_p4["min_univ"] - proj_p4["max_inter"]:+.4f}')

## PART 2 — Lewis game with production cost

In [ ]:
print('\n' + '═'*60)
print('PART 2 — Lewis Game: no cost vs production cost')
print('═'*60)

N_SYMBOLS = len(CONCEPTS)        # vocabulary = number of proto-tokens
MSG_LEN   = 2
EMB_DIM   = 64

class LewisSender(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(768, 128), nn.ReLU())
        self.heads = nn.ModuleList([nn.Linear(128, N_SYMBOLS) for _ in range(MSG_LEN)])
    def forward(self, x):
        h = self.net(x)
        return [head(h) for head in self.heads]

class LewisReceiver(nn.Module):
    def __init__(self):
        super().__init__()
        self.sym_emb = nn.ModuleList([nn.Embedding(N_SYMBOLS, EMB_DIM) for _ in range(MSG_LEN)])
        self.out = nn.Sequential(nn.Linear(EMB_DIM*MSG_LEN, 128), nn.ReLU(), nn.Linear(128, 768))
    def forward(self, toks):
        embs = [self.sym_emb[i](toks[i]) for i in range(MSG_LEN)]
        h = torch.cat(embs, dim=-1)
        return F.normalize(self.out(h), dim=-1)

def train_lewis(production_cost=0.0, n_epochs=600):
    """Train a Lewis game. production_cost weights an entropy penalty on symbol usage."""
    X = torch.tensor(projected_matrix, dtype=torch.float32)
    sender = LewisSender(); receiver = LewisReceiver()
    opt = torch.optim.Adam(list(sender.parameters())+list(receiver.parameters()), lr=1e-3)
    entropy_history = []
    baseline = 0.0
    for epoch in range(n_epochs):
        logits_list = sender(X)
        dists = [torch.distributions.Categorical(logits=lg) for lg in logits_list]
        toks = [d.sample() for d in dists]
        logps = sum(d.log_prob(t) for d, t in zip(dists, toks))
        recon = receiver(toks)
        reward = (recon * X).sum(-1)              # cosine reward
        # Production cost: penalise high per-symbol entropy (encourage lean vocab)
        usage_entropy = 0.0
        for t in toks:
            counts = torch.bincount(t, minlength=N_SYMBOLS).float() + 1e-6
            p = counts / counts.sum()
            usage_entropy = usage_entropy + (-(p * p.log()).sum())
        cost = production_cost * usage_entropy
        advantage = (reward - baseline).detach()
        loss = -(advantage * logps).mean() - 0.01 * sum(d.entropy().mean() for d in dists) \
               - (reward).mean() + cost
        opt.zero_grad(); loss.backward(); opt.step()
        baseline = 0.9*baseline + 0.1*reward.mean().item()
        # track symbol-usage entropy
        with torch.no_grad():
            t0 = toks[0]
            counts = torch.bincount(t0, minlength=N_SYMBOLS).float()
            p = counts / counts.sum()
            ent = -(p[p>0] * p[p>0].log()).sum().item()
            entropy_history.append(ent)
    # Final symbol usage distribution
    with torch.no_grad():
        logits_list = sender(X)
        toks = [lg.argmax(-1) for lg in logits_list]
        counts = torch.bincount(toks[0], minlength=N_SYMBOLS).float().numpy()
    return entropy_history, counts, float(reward.mean())

print('\nTraining WITHOUT production cost...')
ent_no, counts_no, rew_no = train_lewis(production_cost=0.0)
print(f'  final symbol entropy: {ent_no[-1]:.3f}  reward: {rew_no:.3f}')

print('Training WITH production cost...')
ent_yes, counts_yes, rew_yes = train_lewis(production_cost=0.05)
print(f'  final symbol entropy: {ent_yes[-1]:.3f}  reward: {rew_yes:.3f}')

def zipf_r2(counts):
    """Fit symbol-frequency rank distribution to a power law; return R²."""
    freqs = np.sort(counts[counts > 0])[::-1]
    if len(freqs) < 3: return 0.0
    ranks = np.arange(1, len(freqs)+1)
    lr = linregress(np.log(ranks), np.log(freqs))
    return float(lr.rvalue**2)

r2_no = zipf_r2(counts_no)
r2_yes = zipf_r2(counts_yes)
print(f'\nZipf power-law R²:  no cost = {r2_no:.3f}   with cost = {r2_yes:.3f}')

## Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Probe 4 comparison
ax = axes[0, 0]
labels = ['raw', 'projected']
mins = [raw_p4['min_univ'], proj_p4['min_univ']]
maxs = [raw_p4['max_inter'], proj_p4['max_inter']]
x = np.arange(2); w = 0.35
ax.bar(x - w/2, mins, w, label='min(univ–lang) ρ', color='#1D9E75')
ax.bar(x + w/2, maxs, w, label='max(cross-lang) ρ', color='#E24B4A')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Spearman ρ')
ax.set_title('Probe 4: universal beats all languages?\n(green > red = EMERGENT)')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Symbol entropy over training
ax = axes[0, 1]
ax.plot(ent_no, label='no production cost', color='#E24B4A')
ax.plot(ent_yes, label='with production cost', color='#1D9E75')
ax.set_xlabel('Training epoch'); ax.set_ylabel('Symbol-usage entropy')
ax.set_title('Lewis game symbol entropy\n(rising = anti-efficient; falling = lean vocab)')
ax.legend(); ax.grid(True, alpha=0.3)

# Symbol usage distribution (no cost)
ax = axes[1, 0]
ax.bar(range(len(counts_no)), np.sort(counts_no)[::-1], color='#E24B4A', alpha=0.8)
ax.set_xlabel('Symbol rank'); ax.set_ylabel('Usage count')
ax.set_title(f'Symbol distribution — NO cost\n(Zipf R²={r2_no:.3f}, flat = uniform)')
ax.grid(axis='y', alpha=0.3)

# Symbol usage distribution (with cost)
ax = axes[1, 1]
ax.bar(range(len(counts_yes)), np.sort(counts_yes)[::-1], color='#1D9E75', alpha=0.8)
ax.set_xlabel('Symbol rank'); ax.set_ylabel('Usage count')
ax.set_title(f'Symbol distribution — WITH cost\n(Zipf R²={r2_yes:.3f}, skewed = Zipfian)')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Experiment 9b — Probe 4 Projected + Production-Cost Lewis Game',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp9b_probe4_production_cost.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print('\n' + '═'*60)
print('EXPERIMENT 9b — SUMMARY')
print('═'*60)
print(f'PART 1 — Probe 4:')
print(f'  raw space:       {raw_p4["verdict"]}')
print(f'  projected space: {proj_p4["verdict"]}')
print(f'PART 2 — Production cost:')
print(f'  symbol entropy:  no-cost={ent_no[-1]:.3f} → with-cost={ent_yes[-1]:.3f}')
print(f'  Zipf R²:         no-cost={r2_no:.3f} → with-cost={r2_yes:.3f}')
print()
if ent_yes[-1] < ent_no[-1] and r2_yes > r2_no:
    print('  → Production cost DOES induce a leaner, more Zipfian vocabulary.')
    print('    This completes the circuit: discreteness → opacity → production pressure.')
else:
    print('  → Production cost effect is weak in this run; tune the cost weight.')